# The Plasm Koans

Each koan below has blanks (`__`) to fill in. Run the setup cell once, then run each koan cell:
it runs the koan in its own sandbox, so every koan starts from the same data, and shows ✔ when
it passes, or what to fill in or fix. Edit a cell and run it again until it passes.

Build the koans first: `dotnet build koans/Plasm.Koans`. The setup cell starts a throwaway
PostgreSQL container, or uses the server named by `PLASM_KOANS_PG`.

In [ ]:
#I "../Plasm.Koans/bin/Debug/net11.0"
#r "Microsoft.Extensions.Logging.Abstractions.dll"
#r "Plasm.dll"
#r "Plasm.Postgres.dll"
#r "Npgsql.dll"
#r "Testcontainers.dll"
#r "Testcontainers.PostgreSql.dll"
#r "Plasm.Koans.Core.dll"
#r "Plasm.Koans.dll"
open Plasm
open Plasm.Koans
open Plasm.Koans.Shop
open Plasm.Koans.Notebook

start ()

## Getting started

A koan is a few lines of F# with a blank (__) to fill in. This lesson shows how a blank works, and the F# the course relies on: values, lists and the pipe, records and with, wrappers, options, results and tasks.

In [ ]:
open Plasm.Koans
open Plasm.Koans.Shop

In [ ]:
koan "Fill in the blank" (fun _ ->
    // A koan is a few lines of F# with a blank, written __. Replace
    // the blank with a value and run the koans again.
    // `actual |> shouldBe expected` passes when the two are equal.
    let name = "Widget"
    name |> shouldBe __)

In [ ]:
koan "Values do not change" (fun _ ->
    // let names a value. The name keeps that value: working out a
    // new value gives a new name, and the old one is unchanged.
    let price = 10m
    let discounted = price - 2m
    discounted |> shouldBe __
    price |> shouldBe __)

In [ ]:
koan "The pipe passes a value along" (fun _ ->
    // [ ... ] is a list. The pipe, |>, hands the value on its left
    // to the function on its right: `xs |> List.sort` is `List.sort xs`.
    let totals = [ 30m; 12.5m; 99m ]
    totals |> List.sort |> List.head |> shouldBe __)

In [ ]:
koan "A record groups named fields" (fun _ ->
    // A record is a value with named fields. Product is a record
    // defined in Shop.fs, with an Id, a Sku and a Name.
    let widget = { Id = ProductId 1L; Sku = "WIDGET"; Name = "Widget" }
    widget.Sku |> shouldBe __)

In [ ]:
koan "with makes a changed copy" (fun _ ->
    // Records cannot be changed. `{ r with Field = value }` makes a
    // new record with that field changed and every other field copied.
    let widget = { Id = ProductId 1L; Sku = "WIDGET"; Name = "Widget" }
    let renamed = { widget with Name = "Blue widget" }
    renamed.Name |> shouldBe __
    widget.Name |> shouldBe __)

In [ ]:
koan "A wrapper keeps a value's meaning" (fun _ ->
    // `type CustomerId = CustomerId of int64` (in Shop.fs) wraps a
    // number, so a customer's id cannot be mixed up with an order's.
    // Matching on the case takes the number back out.
    let ada = CustomerId 1L
    let (CustomerId n) = ada
    n |> shouldBe __)

In [ ]:
koan "An option may be absent" (fun _ ->
    // An option is either Some value or None (nothing there).
    // Option.defaultValue gives the value, or the default for None.
    let reason: string option = Some "duplicate"
    let noReason: string option = None
    reason |> Option.defaultValue "no reason" |> shouldBe __
    noReason |> Option.defaultValue "no reason" |> shouldBe __)

In [ ]:
koan "A result is Ok or Error" (fun _ ->
    // A Result is either Ok value or Error reason. Quantity.create
    // (in Shop.fs) refuses a quantity that is not positive.
    // `ok r` takes the value out of Ok, `err r` the reason out of Error;
    // given the other case, they fail the koan.
    let three = Quantity.create 3
    let zero = Quantity.create 0
    (ok three).Value |> shouldBe __
    err zero |> shouldBe __)

In [ ]:
koanTask "let! waits for a task" (fun _ ->
    // A task is work that finishes later, such as a database call.
    // Inside task { ... }, let! waits for a task and names its result.
    task {
        let countLater () = task { return 2 + 2 }
        let! n = countLater ()
        n |> shouldBe __
    })

## Meet the shop

The koans database is a small shop: customers place orders, each order has lines, and each line is for a product. Every koan starts from the same data, and its changes vanish when it ends.

In [ ]:
open Plasm
open Plasm.Koans
open Plasm.Koans.Shop

In [ ]:
koanTask "The shop has three customers" (fun k ->
    task {
        // k.Repo reads and writes the koans database (lesson 03 says
        // what a repo is). Query.from Db.customers means "every row of
        // the customers table"; All returns them as Customer records.
        let! customers = k.Repo.All(Query.from Db.customers)
        let names = ok customers |> List.map (fun c -> c.Name) |> List.sort
        names |> shouldBe __
    })

In [ ]:
koanTask "Counting rows" (fun k ->
    task {
        // Count asks the database how many rows there are, without
        // reading them. The starting data has three products.
        let! products = k.Repo.Count(Query.from Db.products)
        ok products |> shouldBe __
    })

In [ ]:
koanTask "The database numbers the rows" (fun k ->
    task {
        // Every table's id is generated by the database: 1, 2, 3 ...
        // in the order the starting data inserts the rows. The id
        // comes back wrapped in its own type (CustomerId, in Shop.fs).
        let! customers = k.Repo.All(Query.from Db.customers)
        let bob = ok customers |> List.find (fun c -> c.Name = "Bob")
        bob.Id |> shouldBe __
    })

In [ ]:
koanTask "An order belongs to a customer" (fun k ->
    task {
        // An order's CustomerId says who placed it: the
        // customer_id column of the orders table.
        let! customers = k.Repo.All(Query.from Db.customers)
        let! orders = k.Repo.All(Query.from Db.orders)
        let ada = ok customers |> List.find (fun c -> c.Name = "Ada")
        let adas = ok orders |> List.filter (fun o -> o.CustomerId = Some ada.Id)
        adas.Length |> shouldBe __
    })

In [ ]:
koanTask "An order may have no customer" (fun k ->
    task {
        // customer_id may be null, so CustomerId is an option.
        let! orders = k.Repo.All(Query.from Db.orders)
        let fourth = ok orders |> List.find (fun o -> o.Id = OrderId 4L)
        fourth.CustomerId |> shouldBe __
    })

In [ ]:
koanTask "A koan's changes stay in the koan" (fun k ->
    task {
        // Every koan runs in a sandbox: a transaction that is rolled
        // back when the koan ends. Inside it, changes are real.
        // (Changeset.insert describes a new row; lesson 10 explains it.)
        let! _ = k.Repo.Insert(Changeset.insert (New.customer "Dan" "dan@example.com"))
        let! customers = k.Repo.Count(Query.from Db.customers)
        ok customers |> shouldBe __
    })

In [ ]:
koanTask "The next koan starts afresh" (fun k ->
    task {
        // The previous koan inserted Dan. This koan has its own sandbox.
        let! customers = k.Repo.Count(Query.from Db.customers)
        ok customers |> shouldBe __
    })

## What a repo is

A repo runs queries and writes against one database with one model. A query is only a description: the repo turns it into SQL, sends it when asked, and returns a Result.

In [ ]:
open Plasm
open Plasm.Koans
open Plasm.Koans.Shop

In [ ]:
koanTask "A repo pairs a model with a database" (fun k ->
    task {
        // A repo (repository) is where Plasm meets the database. It is
        // made from a model, the record types Plasm can store (Db.model,
        // built from the schemas in Shop.fs), and a database connection.
        k.Repo.Model.Types.Length |> shouldBe __

        // CheckSchema compares the model with the database's tables
        // and lists every difference it finds.
        let! issues = k.Repo.CheckSchema()
        ok issues |> shouldBe __
    })

In [ ]:
koan "A query is a description" (fun k ->
    // A query is a value that describes what to read. Building it
    // sends nothing. k.Commands lists the SQL this koan has sent.
    let _customers = Query.from Db.customers
    k.Commands.Length |> shouldBe __)

In [ ]:
koan "The repo shows its SQL" (fun k ->
    // A repo turns a query into SQL. k.Sql shows the SQL it would
    // send, without sending it.
    let products = Query.from Db.products
    let sql = k.Sql products
    sql.Contains "FROM \"products\"" |> shouldBe __
    k.Commands.Length |> shouldBe __)

In [ ]:
koanTask "Running sends that SQL" (fun k ->
    task {
        // The repo sends a query when you run it, with All, One or Count.
        let products = Query.from Db.products
        let sql = k.Sql products
        let! _ = k.Repo.All products
        k.Commands.Length |> shouldBe __
        (k.Commands.Head = sql) |> shouldBe __
    })

In [ ]:
koanTask "A query runs each time you ask" (fun k ->
    task {
        // A repo does not remember results: each run reads the
        // database again. The query itself is unchanged by running it.
        let products = Query.from Db.products
        let! first = k.Repo.All products
        let! second = k.Repo.All products
        k.Commands.Length |> shouldBe __
        (ok first = ok second) |> shouldBe __
    })

In [ ]:
koanTask "Every call returns a Result" (fun k ->
    task {
        // Every repo call returns a Result: Ok with what it read, or
        // Error with what went wrong. One returns an option: the row,
        // or None. `plasm { ... where ... }` keeps only matching rows.
        let! orders = k.Repo.Count(Query.from Db.orders)
        let! zed = k.Repo.One(plasm { for c in Db.customers do where (c.Name = "Zed") })
        orders |> shouldBe __
        zed |> shouldBe __
    })

In [ ]:
koanTask "A repo knows only its model" (fun k ->
    task {
        // A model that knows only products. k.RepoWith makes a repo
        // over it, on the same database.
        let productsOnly = Model.buildOrFail Db.scalars [ Db.products ]
        let! customers = (k.RepoWith productsOnly).All(Query.from Db.customers)

        // It cannot read customers: it returns an Error, it does not throw.
        match err customers with
        | DbError.Translation(TranslationError.UnmappedType name) -> name |> shouldBe __
        | other -> failwithf "expected an unmapped type, got %A" other

        k.Commands.Length |> shouldBe __
    })

## Plasm's promises

Plasm keeps a few promises everywhere: plain records, nothing run until asked, no hidden or in-memory work, failures as values, minimal writes, a checked model, F# meaning, batched loads and tests that roll back. Each koan shows one and names the lesson that goes deeper.

In [ ]:
open Plasm
open Plasm.Koans
open Plasm.Koans.Shop

In [ ]:
koanTask "Records stay plain F#" (fun k ->
    task {
        // A record read from the database is an ordinary F# value: it
        // is not watched, and changing a copy writes nothing.
        // (Lesson 05 goes deeper.)
        let! products = k.Repo.All(Query.from Db.products)
        let widget = ok products |> List.find (fun p -> p.Sku = "WIDGET")
        let _renamed = { widget with Name = "Blue widget" }
        widget.Name |> shouldBe __
        k.Commands.Length |> shouldBe __
    })

In [ ]:
koan "Nothing runs until you ask" (fun k ->
    // A query and a changeset (a description of a write) are values.
    // Only a repo call sends anything. (Lessons 06 and 10 go deeper.)
    let _drafts = plasm { for o in Db.orders do where (o.Status = Draft) }
    let _dan = Changeset.insert (New.customer "Dan" "dan@example.com")
    k.Commands.Length |> shouldBe __)

In [ ]:
koanTask "No hidden queries" (fun k ->
    task {
        // An order's Lines is a relationship: Loaded with the lines, or
        // NotLoaded. Reading a NotLoaded relationship never queries the
        // database behind your back. (Lesson 09 goes deeper.)
        let! orders = k.Repo.All(Query.from Db.orders)
        let first = ok orders |> List.find (fun o -> o.Id = OrderId 1L)
        first.Lines |> shouldBe __
        k.Commands.Length |> shouldBe __
    })

In [ ]:
koanTask "Nothing is evaluated behind your back" (fun k ->
    task {
        // shout is ordinary F#, which SQL cannot run. Plasm refuses the
        // query rather than reading every customer to run it in memory.
        // (Lesson 07 goes deeper.)
        let shout (s: string) = s.ToUpperInvariant() + "!"
        let! r = k.Repo.All(plasm { for c in Db.customers do where (shout c.Name = "ADA!") })
        Result.isError r |> shouldBe __
        k.Commands.Length |> shouldBe __
    })

In [ ]:
koanTask "A failure is a value" (fun k ->
    task {
        // Ada's email is taken, and the customers table's email has a
        // unique constraint (see the schema). The failure comes back as
        // an Error you can match on; nothing is thrown. (Lesson 20 goes deeper.)
        let! r = k.Repo.Insert(Changeset.insert (New.customer "Ada again" "ada@example.com"))

        match err r with
        | WriteError.Failed failure -> failure.Constraint |> shouldBe __
        | other -> failwithf "expected a database failure, got %A" other
    })

In [ ]:
koanTask "Only what you change is written" (fun k ->
    task {
        // Changeset.edit describes a change to a record you hold. The
        // UPDATE sets only the fields that changed. (Lesson 11 goes deeper.)
        let! drafts = k.Repo.All(plasm { for o in Db.orders do where (o.Status = Draft) })
        let draft = ok drafts |> List.head
        let! _ = k.Repo.Update(draft |> Changeset.edit (fun o -> { o with Status = Published }))
        let update = List.last k.Commands
        update.Contains "\"status\" = " |> shouldBe __
        update.Contains "\"total\" = " |> shouldBe __
    })

In [ ]:
koan "The model is checked before any query" (fun k ->
    // Model.build checks every schema once, when the model is made.
    // Without scalar mappings (Db.scalars), it cannot store a ProductId.
    // Each error names the record and the field (Property) at fault.
    // (Lesson 05 goes deeper.)
    match Model.build [] [ Db.products ] with
    | Ok _ -> failwith "expected a configuration error"
    | Error errors ->
        (errors |> List.map (fun e -> e.Property) |> List.distinct) |> shouldBe __
        k.Commands.Length |> shouldBe __)

In [ ]:
koanTask "Queries mean what F# means" (fun k ->
    task {
        // In F#, `o.CustomerId = None` is true for an order with no
        // customer. In the query it means the same: SQL's IS NULL.
        // (Lesson 07 goes deeper.)
        let noCustomer = plasm { for o in Db.orders do where (o.CustomerId = None) }
        (k.Sql noCustomer).Contains "IS NULL" |> shouldBe __
        let! n = k.Repo.Count noCustomer
        ok n |> shouldBe __
    })

In [ ]:
koanTask "Relationships load in batches" (fun k ->
    task {
        // preload asks for a relationship to be loaded with the rows.
        // The lines of all four orders come in one command, not one
        // command per order. (Lesson 09 goes deeper.)
        let! _ = k.Repo.All(plasm { for o in Db.orders do preload o.Lines })
        k.Commands.Length |> shouldBe __
    })

In [ ]:
koanTask "Tests roll back" (fun k ->
    task {
        // repo.Sandbox runs code in a transaction that is rolled back
        // when it ends: every koan runs in one. (Lesson 21 goes deeper.)
        let! inside =
            k.Repo.Sandbox(fun repo ->
                task {
                    let! _ = repo.Insert(Changeset.insert (New.customer "Dan" "dan@example.com"))
                    return! repo.Count(Query.from Db.customers)
                })

        let! after = k.Repo.Count(Query.from Db.customers)
        ok inside |> shouldBe __
        ok after |> shouldBe __
    })

## About records and schemas

Domain types are ordinary immutable F# records and single-case unions. Persistence lives in separate schema values and scalar mappings, and a model is validated once, before any query runs.

In [ ]:
open Plasm
open Plasm.Koans
open Plasm.Koans.Shop

In [ ]:
koan "A record needs nothing from Plasm" (fun _ ->
    // Product is a plain F# record: no base class, no attributes.
    // Its table and key are described separately, in Db.products.
    let widget = { Id = ProductId 1L; Sku = "WIDGET"; Name = "Widget" }
    let renamed = { widget with Name = "Better widget" }
    widget.Name |> shouldBe __
    renamed.Name |> shouldBe __)

In [ ]:
koan "A schema names the table" (fun k ->
    // A schema is a value that says how a record is stored. In Shop.fs,
    // Db.products is: schema<Product> "products" { identityKey (fun p -> p.Id) }
    // Query.from makes a query for every row of a schema's table.
    // k.Sql shows the SQL a query would send, without sending it.
    let sql = k.Sql(Query.from Db.products)
    // Which table does the query read from?
    sql.Contains "FROM \"products\"" |> shouldBe __)

In [ ]:
koan "Columns are named in snake_case" (fun k ->
    // A column is named after its field, in snake_case: lower case words
    // joined by underscores. The schema could name a column itself, but
    // Db.orders does not.
    let sql = k.Sql(Query.from Db.orders)
    // Order has a field PlacedAt. Which name does the SQL use?
    sql.Contains "\"placed_at\"" |> shouldBe __
    sql.Contains "\"PlacedAt\"" |> shouldBe __)

In [ ]:
koanTask "A key finds one record" (fun k ->
    task {
        // The key is the field that identifies a record: identityKey (fun p -> p.Id).
        // The database numbers new rows 1, 2, 3, in the order they were inserted.
        // A repo (k.Repo) runs queries against the database; One reads one record, if any.
        let! product = k.Repo.One(plasm { for p in Db.products do where (p.Id = ProductId 2L) })
        // ok takes the value out of an Ok result; .Value takes it out of Some.
        (ok product).Value.Name |> shouldBe __
    })

In [ ]:
koanTask "A union is stored as text" (fun k ->
    task {
        // OrderStatus is a union with no fields. Scalar.enum in Db.scalars stores
        // each case as text: Draft as "draft", Published as "published", and so on.
        // Here the status column of Order 2 is changed with SQL, as text.
        // (The change is made in this koan's sandbox and vanishes when it ends.)
        let! _ = k.Repo.ExecuteSql $"update orders set status = 'cancelled' where id = 2"
        let! order = k.Repo.One(plasm { for o in Db.orders do where (o.Id = OrderId 2L) })
        // Is Status the text, or a case of OrderStatus?
        (ok order).Value.Status |> shouldBe __
    })

In [ ]:
koanTask "Wrapped keys keep their meaning" (fun k ->
    task {
        // CustomerId wraps the bigint key (Scalar.wrap in Db.scalars).
        let! ada = k.Repo.One(plasm { for c in Db.customers do where (c.Email = "ada@example.com") })
        let ada = (ok ada).Value
        ada.Id |> shouldBe __
    })

In [ ]:
koanTask "Stored values can be rejected" (fun k ->
    task {
        // Quantity can only be built by Quantity.create, which refuses zero.
        Quantity.create 0 |> Result.isOk |> shouldBe __

        // A zero stored in the database is refused on the way in, with its field named.
        let! _ = k.Repo.ExecuteSql $"update order_lines set quantity = 0 where id = 1"
        let! lines = k.Repo.All(Query.from Db.orderLines)

        match lines with
        | Error(DbError.Materialization m) -> m.Property |> shouldBe __
        | other -> failwithf "expected a materialisation error, got %A" other
    })

In [ ]:
koanTask "Absent values are options" (fun k ->
    task {
        // Order 4 has no customer (a NULL column) but has a cancellation reason.
        let! order = k.Repo.One(plasm { for o in Db.orders do where (o.Id = OrderId 4L) })
        let order = (ok order).Value
        order.CustomerId |> shouldBe __
        order.CancellationReason |> shouldBe __
    })

In [ ]:
koan "A model reports every mistake at once" (fun _ ->
    // Building a model with no scalar mappings: CustomerId, OrderId,
    // OrderStatus and the rest cannot be stored.
    match Model.build [] [ Db.customers; Db.orders ] with
    | Ok _ -> failwith "expected configuration errors"
    | Error errors ->
        (errors.Length > 1) |> shouldBe __
        errors |> List.exists (fun e -> e.Model = "Order") |> shouldBe __)

## About query values

A query is a value: building, extending and inspecting it never touches the database. Execution happens only at the repository.

In [ ]:
open Plasm
open Plasm.Koans
open Plasm.Koans.Shop

In [ ]:
koanTask "Query.from reads every row" (fun k ->
    task {
        // A query describes which rows to read. Query.from is the plainest
        // query there is: every row of a schema's table, as it is.
        // k.Repo.All runs a query and returns all the rows it matches.
        let! customers = k.Repo.All(Query.from Db.customers)
        // How many customers does the koans database start with?
        (ok customers).Length |> shouldBe __
    })

In [ ]:
koanTask "A where keeps the matching rows" (fun k ->
    task {
        // plasm { } writes a query step by step. `for o in Db.orders do` names
        // each Order o; `where` keeps the rows for which its condition is true.
        let drafts = plasm { for o in Db.orders do where (o.Status = Draft) }
        let! orders = k.Repo.All drafts
        // How many Orders are drafts?
        (ok orders).Length |> shouldBe __
    })

In [ ]:
koanTask "An orderBy sorts the rows" (fun k ->
    task {
        // orderBy sorts the rows by a field, smallest first (for text: A to Z).
        // The database does the sorting.
        let byName = plasm { for p in Db.products do orderBy p.Name }
        let! products = k.Repo.All byName
        ok products |> List.map (fun p -> p.Name) |> shouldBe __
    })

In [ ]:
koanTask "take keeps the first rows" (fun k ->
    task {
        // take n keeps the first n rows. "First" needs an order, so the
        // query sorts the Customers by name before taking two.
        let firstTwo = plasm { for c in Db.customers do orderBy c.Name; take 2 }
        let! customers = k.Repo.All firstTwo
        ok customers |> List.map (fun c -> c.Name) |> shouldBe __
    })

In [ ]:
koan "Building a query runs nothing" (fun k ->
    let _published = plasm { for o in Db.orders do where (o.Status = Published) }
    // How many commands has this koan sent?
    k.Commands.Length |> shouldBe __)

In [ ]:
koanTask "Extending makes a new query" (fun k ->
    task {
        let all = Query.from Db.orders
        let published = plasm { for o in all do where (o.Status = Published) }
        let! everyOrder = k.Repo.Count all
        let! publishedOrders = k.Repo.Count published
        ok everyOrder |> shouldBe __
        ok publishedOrders |> shouldBe __
    })

In [ ]:
koanTask "Running does not change a query" (fun k ->
    task {
        let drafts = plasm { for o in Db.orders do where (o.Status = Draft) }
        let! first = k.Repo.All drafts
        let! second = k.Repo.All drafts
        (ok first = ok second) |> shouldBe __
        (ok first).Length |> shouldBe __
    })

In [ ]:
koan "A query can explain itself" (fun k ->
    let published = plasm { for o in Db.orders do where (o.Status = Published) }
    let sql = k.Sql published
    // Is the status sent as a parameter?
    sql.Contains "\"status\" = @p0" |> shouldBe __
    k.Commands.Length |> shouldBe __)

In [ ]:
koanTask "Reusable filters are functions" (fun k ->
    task {
        let published (q: Query<Order>) = plasm { for o in q do where (o.Status = Published) }
        let over (amount: decimal) (q: Query<Order>) = plasm { for o in q do where (o.Total > amount) }
        let! n = k.Repo.Count(Query.from Db.orders |> published |> over 50m)
        ok n |> shouldBe __
    })

In [ ]:
koanTask "Captured values become parameters" (fun k ->
    task {
        let email = "bob@example.com"
        let byEmail = plasm { for c in Db.customers do where (c.Email = email) }
        // Is the email written into the SQL text?
        (k.Sql byEmail).Contains "bob@example.com" |> shouldBe __
        let! bob = k.Repo.One byEmail
        (ok bob).Value.Name |> shouldBe __
    })

## About filters and ordering

Filters mean what the same F# predicate means. Anything that cannot run in the database is refused, never evaluated in memory.

In [ ]:
open Plasm
open Plasm.Koans
open Plasm.Koans.Shop

let count (k: KoanContext) (q: Query<Order>) =
    task {
        let! n = k.Repo.Count q
        return ok n
    }

In [ ]:
koanTask "Comparing numbers" (fun k ->
    task {
        // A filter compares a field as F# would: Total is a decimal, so it is
        // compared with a decimal (20m). count, defined at the top of this
        // lesson, runs k.Repo.Count: how many rows the query matches.
        let! n = count k (plasm { for o in Db.orders do where (o.Total > 20m) })
        n |> shouldBe __
    })

In [ ]:
koanTask "Contains on text becomes LIKE" (fun k ->
    task {
        // s.Contains "o" is true when the text s has an "o" in it. In a filter it
        // becomes SQL's LIKE, which matches text against a pattern.
        let withO = plasm { for c in Db.customers do where (c.Name.Contains "o") }
        (k.Sql withO).Contains "LIKE" |> shouldBe __
        let! customers = k.Repo.All withO
        ok customers |> List.map (fun c -> c.Name) |> shouldBe __
    })

In [ ]:
koanTask "And needs both" (fun k ->
    task {
        // && keeps a row only when both sides are true.
        let! n = count k (plasm { for o in Db.orders do where (o.Status = Published && o.Total < 50m) })
        n |> shouldBe __
    })

In [ ]:
koanTask "Or needs either" (fun k ->
    task {
        // || keeps a row when either side is true, or both.
        let! n = count k (plasm { for o in Db.orders do where (o.Status = Draft || o.Total > 50m) })
        n |> shouldBe __
    })

In [ ]:
koanTask "Ordering largest first" (fun k ->
    task {
        // orderByDescending sorts the rows largest first.
        let! orders = k.Repo.All(plasm { for o in Db.orders do orderByDescending o.Total })
        ok orders |> List.map (fun o -> o.Total) |> shouldBe __
    })

In [ ]:
koanTask "Independent filters combine" (fun k ->
    task {
        let! n =
            count k (plasm {
                for o in Db.orders do
                    where (o.Status = Published)
                    where (o.Total > 50m)
            })

        n |> shouldBe __
    })

In [ ]:
koanTask "Comparing with None" (fun k ->
    task {
        let withoutCustomer = plasm { for o in Db.orders do where (o.CustomerId = None) }
        (k.Sql withoutCustomer).Contains "IS NULL" |> shouldBe __
        let! n = count k withoutCustomer
        n |> shouldBe __
    })

In [ ]:
koanTask "not keeps F#'s meaning" (fun k ->
    task {
        // Orders that are not Ada's: in F#, an Order with no customer is not Ada's.
        let notAdas = plasm { for o in Db.orders do where (not (o.CustomerId = Some(CustomerId 1L))) }
        let! n = count k notAdas
        n |> shouldBe __
    })

In [ ]:
koanTask "Membership in an empty list" (fun k ->
    task {
        let wanted: OrderStatus list = []
        let! n = count k (plasm { for o in Db.orders do where (List.contains o.Status wanted) })
        n |> shouldBe __
    })

In [ ]:
koanTask "Some code cannot be translated" (fun k ->
    task {
        let shout (s: string) = s.ToUpperInvariant() + "!"
        let! r = k.Repo.All(plasm { for c in Db.customers do where (shout c.Name = "ADA!") })

        let refused =
            match r with
            | Error(DbError.Translation(TranslationError.UnsupportedExpression _)) -> true
            | _ -> false

        refused |> shouldBe __
    })

In [ ]:
koanTask "Pages need an order" (fun k ->
    task {
        let! unordered = k.Repo.All(plasm { for o in Db.orders do take 2 })
        Result.isError unordered |> shouldBe __

        let! page = k.Repo.All(plasm { for o in Db.orders do orderBy o.Total; take 2 })
        ok page |> List.map (fun o -> o.Total) |> shouldBe __
    })

In [ ]:
koanTask "Captured arrays are read each run" (fun k ->
    task {
        let statuses = [| Published |]
        let matching = plasm { for o in Db.orders do where (Array.contains o.Status statuses) }
        let! before = count k matching
        statuses.[0] <- Draft
        let! after = count k matching
        before |> shouldBe __
        after |> shouldBe __
    })

## About reading

The repository runs queries and returns Results. Counting and existence transfer no rows; projections read only the columns they use.

In [ ]:
open Plasm
open Plasm.Koans
open Plasm.Koans.Shop

In [ ]:
koanTask "All returns a list" (fun k ->
    task {
        // k.Repo.All returns a Result: Ok with an F# list of the matching
        // records, or Error saying what went wrong. ok takes the list out.
        let! products = k.Repo.All(plasm { for p in Db.products do orderBy p.Id })
        let products: Product list = ok products
        products |> List.map (fun p -> p.Sku) |> shouldBe __
    })

In [ ]:
koanTask "One returns an option" (fun k ->
    task {
        // k.Repo.One reads at most one record. Its value is an option:
        // Some record when a row matches, None when none does.
        let! cy = k.Repo.One(plasm { for c in Db.customers do where (c.Email = "cy@example.com") })
        ok cy |> Option.map (fun c -> c.Name) |> shouldBe __
    })

In [ ]:
koanTask "Count returns an int64" (fun k ->
    task {
        // k.Repo.Count returns how many rows match, as an int64
        // (a 64-bit integer, written with an L: 7L).
        let! n = k.Repo.Count(Query.from Db.products)
        ok n |> shouldBe __
    })

In [ ]:
koanTask "Selecting one field" (fun k ->
    task {
        // select says what to read for each row: here only the Name,
        // so the result is a list of strings, not of Customers.
        let! names = k.Repo.All(plasm { for c in Db.customers do orderBy c.Id; select c.Name })
        ok names |> shouldBe __
    })

In [ ]:
koanTask "Reading returns typed rows" (fun k ->
    task {
        let! orders = k.Repo.All(plasm { for o in Db.orders do orderBy o.Id })
        let first = (ok orders).Head
        // Total is a decimal; Status is an OrderStatus, not a string.
        first.Total |> shouldBe __
        first.Status |> shouldBe __
    })

In [ ]:
koanTask "Counting transfers no rows" (fun k ->
    task {
        let! n = k.Repo.Count(plasm { for o in Db.orders do where (o.Status = Published) })
        ok n |> shouldBe __
        k.Commands.Head.Contains "COUNT(*)" |> shouldBe __
    })

In [ ]:
koanTask "Asking whether anything exists" (fun k ->
    task {
        let! any = k.Repo.Exists(plasm { for o in Db.orders do where (o.Total > 1000m) })
        ok any |> shouldBe __
    })

In [ ]:
koanTask "Selecting only what you need" (fun k ->
    task {
        let totals = plasm { for o in Db.orders do orderBy o.Id; select (o.Id, o.Total) }
        let! rows = k.Repo.All totals
        (ok rows).Head |> shouldBe __
        // Is the status column read?
        k.Commands.Head.Contains "\"status\"" |> shouldBe __
    })

In [ ]:
koanTask "Leaving the database is explicit" (fun k ->
    task {
        let! published = k.Repo.All(plasm { for o in Db.orders do where (o.Status = Published) })
        // From here on it is an ordinary F# list, in memory.
        ok published |> List.sumBy (fun o -> o.Total) |> shouldBe __
        k.Commands.Length |> shouldBe __
    })

In [ ]:
koanTask "One means at most one" (fun k ->
    task {
        let! largest = k.Repo.One(plasm { for o in Db.orders do orderByDescending o.Total; take 1 })
        (ok largest |> Option.map (fun o -> o.Total)) |> shouldBe __
        let! none = k.Repo.One(plasm { for o in Db.orders do where (o.Total > 1000m) })
        ok none |> shouldBe __
    })

## About relationships

Relationships are loaded only when asked, in batched commands, and never by reading a field.

In [ ]:
open Plasm
open Plasm.Koans
open Plasm.Koans.Shop

let byId (orders: Order list) (id: int64) = orders |> List.find (fun o -> o.Id = OrderId id)

In [ ]:
koanTask "Preloading a belongs-to" (fun k ->
    task {
        // A relationship links records: each Order belongs to a Customer
        // (belongsTo in Db.orders, through the CustomerId field).
        // A relationship field is an Assoc: NotLoaded, or Loaded with its value.
        // preload asks the query to load it. byId, at the top of this lesson,
        // finds the Order with a given id in a list.
        let! orders = k.Repo.All(plasm { for o in Db.orders do preload o.Customer })
        let order = byId (ok orders) 3L
        // An Order's Customer is optional, so the loaded value is an option.
        order.Customer |> Assoc.map (Option.map (fun c -> c.Name)) |> shouldBe __
    })

In [ ]:
koanTask "Preloading a has-many" (fun k ->
    task {
        // An Order has many OrderLines (hasMany in Db.orders): Lines is a list.
        let! orders = k.Repo.All(plasm { for o in Db.orders do preload o.Lines })
        let order = byId (ok orders) 1L
        order.Lines |> Assoc.map List.length |> shouldBe __
        // Payments was not asked for.
        order.Payments |> shouldBe __
    })

In [ ]:
koanTask "An Order without a Customer is kept" (fun k ->
    task {
        // Order 4 has no customer. Loading Customers does not drop it.
        let! orders = k.Repo.All(plasm { for o in Db.orders do preload o.Customer })
        (ok orders).Length |> shouldBe __
        (byId (ok orders) 4L).Customer |> shouldBe __
    })

In [ ]:
koanTask "Reading a loaded relationship" (fun k ->
    task {
        let! orders = k.Repo.All(plasm { for o in Db.orders do preload o.Lines })
        let order = byId (ok orders) 2L
        // Assoc.get takes the value out of Loaded. It never queries: on a
        // NotLoaded relationship it fails, saying the relationship was not loaded.
        let lines = order.Lines |> Assoc.get
        lines.Head.UnitPrice |> shouldBe __
    })

In [ ]:
koanTask "Not loaded is not empty" (fun k ->
    task {
        let! plain = k.Repo.All(Query.from Db.orders)
        (byId (ok plain) 1L).Lines |> shouldBe __

        // Order 4 has no lines: once loaded, that is an empty list.
        let! loaded = k.Repo.All(plasm { for o in Db.orders do preload o.Lines })
        (byId (ok loaded) 4L).Lines |> shouldBe __
    })

In [ ]:
koanTask "Reading a field never queries" (fun k ->
    task {
        let! orders = k.Repo.All(Query.from Db.orders)

        for o in ok orders do
            for _ in 1..3 do
                o.Lines |> ignore

        k.Commands.Length |> shouldBe __
    })

In [ ]:
koanTask "Loads are batched across parents" (fun k ->
    task {
        // Four Orders, and their Lines.
        let! _ = k.Repo.All(plasm { for o in Db.orders do preload o.Lines })
        k.Commands.Length |> shouldBe __
    })

In [ ]:
koanTask "Siblings load separately" (fun k ->
    task {
        let! orders = k.Repo.All(plasm { for o in Db.orders do preload o.Lines; preload o.Payments })
        let first = byId (ok orders) 1L
        k.Commands.Length |> shouldBe __
        (first.Lines |> Assoc.get |> List.length, first.Payments |> Assoc.get |> List.length) |> shouldBe __
    })

In [ ]:
koanTask "Loads nest" (fun k ->
    task {
        let lines = plasm { for l in Db.orderLines do orderBy l.Id; preload l.Product }
        let! orders = k.Repo.All(plasm { for o in Db.orders do preload o.Lines lines })
        let firstLine = (byId (ok orders) 1L).Lines |> Assoc.get |> List.head
        (firstLine.Product |> Assoc.get).Name |> shouldBe __
        k.Commands.Length |> shouldBe __
    })

In [ ]:
koanTask "A child filter filters children" (fun k ->
    task {
        let succeeded = plasm { for p in Db.payments do where (p.Status = Succeeded) }
        let! orders = k.Repo.All(plasm { for o in Db.orders do preload o.Payments succeeded })
        (ok orders).Length |> shouldBe __
        (byId (ok orders) 3L).Payments |> Assoc.get |> List.length |> shouldBe __
    })

In [ ]:
koanTask "Filtering parents through children" (fun k ->
    task {
        // Order 3 has two payments, one of them succeeded.
        let paid = plasm { for o in Db.orders do where (Rel.any (fun p -> p.Status = Succeeded) o.Payments); orderBy o.Id }
        let! orders = k.Repo.All paid
        ok orders |> List.map (fun o -> o.Id) |> shouldBe __
    })

In [ ]:
koanTask "At most n per parent" (fun k ->
    task {
        let latest = plasm { for p in Db.payments do orderByDescending p.CreatedAt; limitPerParent 1 }
        let! orders = k.Repo.All(plasm { for o in Db.orders do preload o.Payments latest })
        (byId (ok orders) 3L).Payments |> Assoc.get |> List.map (fun p -> p.Status) |> shouldBe __
    })

In [ ]:
koanTask "Loading onto values you hold" (fun k ->
    task {
        let! orders = k.Repo.All(Query.from Db.orders)
        let originals = ok orders
        let! withLines = k.Repo.Preload(plasm { for o in Db.orders do preload o.Lines }, originals)
        (byId originals 1L).Lines |> shouldBe __
        (byId (ok withLines) 1L).Lines |> Assoc.get |> List.length |> shouldBe __
    })

## About changesets

Changes are values that can be validated and inspected before anything is written.

In [ ]:
open System.Collections.Generic
open Plasm
open Plasm.Koans
open Plasm.Koans.Shop

let ada = { New.customer "Ada" "ada@example.com" with Id = CustomerId 1L }

In [ ]:
koan "A changeset starts from a record" (fun _ ->
    // A changeset describes one change to one record; building it writes nothing.
    // Its Action says what saving it will do: ChangeAction.Insert, Update or Delete.
    let adding = Changeset.insert (New.customer "Dan" "dan@example.com")
    let changing = Changeset.update ada
    adding.Action |> shouldBe __
    changing.Action |> shouldBe __)

In [ ]:
koan "Changes are visible before saving" (fun _ ->
    // Changeset.edit makes an update changeset from the fields your function alters.
    let cs = ada |> Changeset.edit (fun c -> { c with Email = "ada@new.example.com" })
    // Proposed is Some record: the original with the changes applied.
    let proposed = cs.Proposed.Value
    proposed.Email |> shouldBe __
    proposed.Name |> shouldBe __)

In [ ]:
koan "A valid changeset has no errors" (fun _ ->
    // A validation checks a proposed value; validateLength checks a string's length.
    let cs = Changeset.insert (New.customer "Dan" "dan@example.com") |> Changeset.validateLength ((fun c -> c.Name), min = 1)
    cs.IsValid |> shouldBe __
    cs.Errors |> shouldBe __)

In [ ]:
koan "A validation adds an error" (fun _ ->
    let cs = Changeset.insert (New.customer "" "dan@example.com") |> Changeset.validateLength ((fun c -> c.Name), min = 1)
    // A failed validation adds a FieldError; its Field names the field it is about.
    cs.IsValid |> shouldBe __
    (List.head cs.Errors).Field |> shouldBe __)

In [ ]:
koan "A changeset leaves its input alone" (fun _ ->
    let cs = ada |> Changeset.edit (fun c -> { c with Name = "Ada L." })
    ada.Name |> shouldBe __
    cs.Changes |> List.map fst |> shouldBe __)

In [ ]:
koan "Unchanged and cleared differ" (fun _ ->
    let cancelled = { New.order None Cancelled 5m with Id = OrderId 4L; CancellationReason = Some "duplicate" }
    let cs = cancelled |> Changeset.edit (fun o -> { o with CancellationReason = None })
    // Every other field is unchanged, so it is not a change at all.
    cs.Changes |> shouldBe __)

In [ ]:
koan "Errors can be inspected" (fun k ->
    let cs =
        Changeset.insert (New.customer "" "not an email")
        |> Changeset.validateLength ((fun c -> c.Name), min = 1)
        |> Changeset.validateFormat ((fun c -> c.Email), "^[^@\\s]+@[^@\\s]+$")

    cs.IsValid |> shouldBe __
    cs.Errors |> List.map (fun e -> e.Field) |> shouldBe __
    k.Commands.Length |> shouldBe __)

In [ ]:
koan "Input cannot touch protected fields" (fun _ ->
    let form = readOnlyDict [ "name", "Mallory"; "email", "ada@new.example.com"; "id", "99" ]
    let cs = Changeset.update ada |> Changeset.cast (form, fun c -> c.Email)
    cs.Changes |> List.map fst |> shouldBe __)

In [ ]:
koanTask "Invalid changes are never written" (fun k ->
    task {
        let invalid = Changeset.insert (New.customer "" "dan@example.com") |> Changeset.validateLength ((fun c -> c.Name), min = 1)
        let! r = k.Repo.Insert invalid

        let isInvalid =
            match r with
            | Error(WriteError.Invalid _) -> true
            | _ -> false

        isInvalid |> shouldBe __
        let! n = k.Repo.Count(Query.from Db.customers)
        ok n |> shouldBe __
    })

In [ ]:
koanTask "Constraints become field errors" (fun k ->
    task {
        let again =
            Changeset.insert (New.customer "Ada again" "ada@example.com")
            |> Changeset.uniqueConstraint ((fun c -> c.Email), "customers_email_key")

        let! r = k.Repo.Insert again

        match r with
        | Error(WriteError.Invalid cs) -> cs.Errors |> List.map (fun e -> e.Field) |> shouldBe __
        | other -> failwithf "expected invalid changes, got %A" other
    })

## About writes

Inserts, updates and deletes take changesets and return new values or typed errors.

In [ ]:
open System
open Plasm
open Plasm.Koans
open Plasm.Koans.Shop

let order (k: KoanContext) (id: int64) =
    task {
        let! o = k.Repo.One(plasm { for o in Db.orders do where (o.Id = OrderId id) })
        return (ok o).Value
    }

In [ ]:
koanTask "Inserting adds a row" (fun k ->
    task {
        // repo.Insert takes an insert changeset, writes it, and returns the stored record.
        let! stored = k.Repo.Insert(Changeset.insert (New.customer "Dan" "dan@example.com"))
        (ok stored).Name |> shouldBe __
        let! n = k.Repo.Count(Query.from Db.customers)
        ok n |> shouldBe __
    })

In [ ]:
koanTask "An insert assigns the key" (fun k ->
    task {
        let! stored = k.Repo.Insert(Changeset.insert (New.customer "Dan" "dan@example.com"))
        // The database chose Dan's key; the stored record carries it, so it finds the row again.
        let danId = (ok stored).Id
        let! found = k.Repo.One(plasm { for c in Db.customers do where (c.Id = danId) })
        (ok found |> Option.map (fun c -> c.Email)) |> shouldBe __
    })

In [ ]:
koanTask "An update returns the new record" (fun k ->
    task {
        let! draft = order k 2L
        // repo.Update takes an update changeset and returns the record as stored.
        let! updated = k.Repo.Update(draft |> Changeset.edit (fun o -> { o with Total = 15m }))
        (ok updated).Total |> shouldBe __
        // Records are values: the one you read is not changed.
        draft.Total |> shouldBe __
    })

In [ ]:
koanTask "The version goes up on update" (fun k ->
    task {
        // Db.orders declares a version field: Version counts the updates of the row.
        let! draft = order k 2L
        let! updated = k.Repo.Update(draft |> Changeset.edit (fun o -> { o with Status = Published }))
        draft.Version |> shouldBe __
        (ok updated).Version |> shouldBe __
    })

In [ ]:
koanTask "Deleting a record" (fun k ->
    task {
        let customers = Query.from Db.customers
        let! cy = k.Repo.One(plasm { for c in Db.customers do where (c.Name = "Cy") })
        // Changeset.delete describes removing a record; only repo.Delete removes it.
        let goodbye = Changeset.delete (ok cy).Value
        let! before = k.Repo.Count customers
        ok before |> shouldBe __
        let! _ = k.Repo.Delete goodbye
        let! after = k.Repo.Count customers
        ok after |> shouldBe __
    })

In [ ]:
koanTask "Inserting returns the stored record" (fun k ->
    task {
        let sprocket = New.product "SPROCKET" "Sprocket"
        let! stored = k.Repo.Insert(Changeset.insert sprocket)
        // The database generated a key (its value depends on earlier inserts).
        ((ok stored).Id <> ProductId 0L) |> shouldBe __
        sprocket.Id |> shouldBe __
    })

In [ ]:
koanTask "Updates write only the changes" (fun k ->
    task {
        let! draft = order k 2L
        let! _ = k.Repo.Update(draft |> Changeset.edit (fun o -> { o with Status = Published }))
        let update = List.last k.Commands
        // Which columns does the UPDATE set?
        update.Contains "\"status\" = " |> shouldBe __
        update.Contains "\"total\" = " |> shouldBe __
    })

In [ ]:
koanTask "Stale updates are refused" (fun k ->
    task {
        let! first = order k 1L
        let! _ = k.Repo.Update(first |> Changeset.edit (fun o -> { o with Total = 31m }))
        // `first` still has the old version.
        let! again = k.Repo.Update(first |> Changeset.edit (fun o -> { o with Total = 32m }))

        let stale =
            match again with
            | Error(WriteError.Stale _) -> true
            | _ -> false

        stale |> shouldBe __
    })

In [ ]:
koanTask "Infrastructure is not a domain error" (fun k ->
    task {
        // orders_total_check is not declared on the changeset, so it stays a database failure.
        let! r = k.Repo.Insert(Changeset.insert (New.order None Draft -1m))

        match r with
        | Error(WriteError.Failed f) -> f.Kind |> shouldBe __
        | other -> failwithf "expected a database failure, got %A" other
    })

In [ ]:
koan "Each call takes its own kind of changeset" (fun k ->
    let refusal =
        try
            k.Repo.Update(Changeset.insert (New.product "X" "X")) |> ignore
            ""
        with :? ArgumentException as e ->
            e.Message

    refusal.StartsWith "repo.Update was given a changeset for insert" |> shouldBe __)

## About child writes

Writing a parent with its children is atomic, and removing children always needs a stated policy and scope.

In [ ]:
open Plasm
open Plasm.Koans
open Plasm.Koans.Shop

let withLines (k: KoanContext) (id: int64) =
    task {
        let! o = k.Repo.One(plasm { for o in Db.orders do where (o.Id = OrderId id); preload o.Lines })
        return (ok o).Value
    }

let linesOf (k: KoanContext) (id: int64) =
    task {
        let! n = k.Repo.Count(plasm { for l in Db.orderLines do where (l.OrderId = OrderId id) })
        return ok n
    }

/// A new draft Order for Cy with one line: two Widgets at 10.00 each.
let cysOrder () =
    let line = Changeset.insert (New.line (ProductId 1L) 2 10m)
    // putChildren adds child changesets (here, Lines) to the parent's changeset.
    Changeset.insert (New.order (Some(CustomerId 3L)) Draft 20m) |> Changeset.putChildren ((fun o -> o.Lines), [ line ])

In [ ]:
koanTask "A parent can be saved with a child" (fun k ->
    task {
        // One repo.Insert writes the Order and then its line.
        let! _ = k.Repo.Insert(cysOrder ())
        let! orders = k.Repo.Count(Query.from Db.orders)
        ok orders |> shouldBe __
        let! lines = k.Repo.Count(Query.from Db.orderLines)
        ok lines |> shouldBe __
    })

In [ ]:
koanTask "The saved parent holds its children" (fun k ->
    task {
        let! saved = k.Repo.Insert(cysOrder ())
        let order = ok saved
        // The returned Order's Lines are loaded with the lines just written.
        let lines = order.Lines |> Assoc.get
        lines.Length |> shouldBe __
        // Each line was given the new Order's key.
        ((List.head lines).OrderId = order.Id) |> shouldBe __
    })

In [ ]:
koanTask "Saved children can be read back" (fun k ->
    task {
        let! saved = k.Repo.Insert(cysOrder ())
        let id = (ok saved).Id
        let! o = k.Repo.One(plasm { for o in Db.orders do where (o.Id = id); preload o.Lines })
        let line = (ok o).Value.Lines |> Assoc.get |> List.head
        line.Quantity.Value |> shouldBe __
        line.UnitPrice |> shouldBe __
    })

In [ ]:
koanTask "Adding a child to an existing parent" (fun k ->
    task {
        // Order 2 has one line. A child changeset with no key is a new line.
        let! o = withLines k 2L
        let added = Changeset.insert (New.line (ProductId 1L) 1 10m)
        let! _ = k.Repo.Update(Changeset.update o |> Changeset.putChildren ((fun o -> o.Lines), [ added ]))
        let! n = linesOf k 2L
        n |> shouldBe __
    })

In [ ]:
koanTask "Omitted children are untouched" (fun k ->
    task {
        let! o = withLines k 1L
        let! _ = k.Repo.Update(o |> Changeset.edit (fun o -> { o with Total = 40m }))
        let! n = linesOf k 1L
        n |> shouldBe __
    })

In [ ]:
koanTask "Children are matched by identity" (fun k ->
    task {
        let! o = withLines k 1L
        let first = o.Lines |> Assoc.get |> List.head
        let repriced = first |> Changeset.edit (fun l -> { l with UnitPrice = 9m })
        let added = Changeset.insert (New.line (ProductId 3L) 1 2.5m)
        let! _ = k.Repo.Update(Changeset.update o |> Changeset.putChildren ((fun o -> o.Lines), [ repriced; added ]))
        let! n = linesOf k 1L
        n |> shouldBe __
    })

In [ ]:
koanTask "Removing children needs a policy" (fun k ->
    task {
        let! o = withLines k 1L
        let keepFirst = [ Changeset.update (o.Lines |> Assoc.get |> List.head) ]
        let replace = Changeset.update o |> Changeset.replaceChildren ((fun o -> o.Lines), ReplaceScope.allChildren, keepFirst)

        // Db.orders declares no OnReplace policy for Lines: removing one is refused.
        let! refused = k.Repo.Update replace
        Result.isError refused |> shouldBe __

        // Db.ordersDeletingLines declares OnReplace.Delete.
        let! allowed = (k.RepoWith Db.modelDeletingLines).Update replace
        ok allowed |> ignore
        let! n = linesOf k 1L
        n |> shouldBe __
    })

In [ ]:
koanTask "Parent and children succeed together" (fun k ->
    task {
        // The second line's negative price breaks order_lines_price_check.
        let lines = [ Changeset.insert (New.line (ProductId 1L) 1 10m); Changeset.insert (New.line (ProductId 2L) 1 -1m) ]
        let! r = k.Repo.Insert(Changeset.insert (New.order None Draft 9m) |> Changeset.putChildren ((fun o -> o.Lines), lines))
        Result.isError r |> shouldBe __
        let! orders = k.Repo.Count(Query.from Db.orders)
        ok orders |> shouldBe __
    })

In [ ]:
koanTask "A filtered list is not the whole list" (fun k ->
    task {
        let! o = withLines k 1L
        let gadgets = plasm { for l in Db.orderLines do where (l.ProductId = ProductId 2L) }
        // Replace only the Gadget lines (with none); the Widget line is outside the scope.
        let replace = Changeset.update o |> Changeset.replaceChildren ((fun o -> o.Lines), ReplaceScope.matching gadgets, [])
        let! _ = (k.RepoWith Db.modelDeletingLines).Update replace
        let! left = k.Repo.All(plasm { for l in Db.orderLines do where (l.OrderId = OrderId 1L) })
        ok left |> List.map (fun l -> l.ProductId) |> shouldBe __
    })

In [ ]:
koanTask "The same child twice is refused" (fun k ->
    task {
        let! o = withLines k 1L
        let first = o.Lines |> Assoc.get |> List.head
        let twice = [ Changeset.update first; Changeset.update first ]
        let! r = k.Repo.Update(Changeset.update o |> Changeset.putChildren ((fun o -> o.Lines), twice))

        let duplicate =
            match r with
            | Error(WriteError.Invalid cs) -> cs.Errors |> List.exists (fun e -> e.Kind = FieldErrorKind.DuplicateIdentity)
            | _ -> false

        duplicate |> shouldBe __
    })

## About transactions

A transaction is a value made of named steps. Running it is explicit; a failure names its step; retries are opt-in.

In [ ]:
open Plasm
open Plasm.Koans
open Plasm.Koans.Shop

let customers (k: KoanContext) =
    task {
        let! n = k.Repo.Count(Query.from Db.customers)
        return ok n
    }

/// A transaction of one step, named "customer", that inserts Dan.
let addDan =
    transaction {
        let! dan = Tx.insert "customer" (Changeset.insert (New.customer "Dan" "dan@example.com"))
        return dan.Name
    }

In [ ]:
koanTask "A transaction does nothing until run" (fun k ->
    task {
        // A transaction is a value describing database steps that succeed or fail together.
        // transaction { } builds it; Tx.insert is a step that inserts a changeset.
        let tx = addDan
        k.Commands.Length |> shouldBe __
        let! n = customers k
        n |> shouldBe __
    })

In [ ]:
koanTask "Running a transaction commits it" (fun k ->
    task {
        // repo.Transact runs the steps and commits them: Ok carries what the transaction returns.
        let! r = k.Repo.Transact addDan
        ok r |> shouldBe __
        let! n = customers k
        n |> shouldBe __
    })

In [ ]:
koanTask "An Error rolls it back" (fun k ->
    task {
        let tx =
            transaction {
                let! _ = Tx.insert "customer" (Changeset.insert (New.customer "Dan" "dan@example.com"))
                // Tx.fail is a step that always fails, with a reason.
                return! Tx.fail "stop" "changed my mind"
            }

        let! r = k.Repo.Transact tx
        Result.isError r |> shouldBe __
        // A failed step rolls back (undoes) the steps before it.
        let! n = customers k
        n |> shouldBe __
    })

In [ ]:
koanTask "A step's result flows to the next" (fun k ->
    task {
        let rename =
            transaction {
                let! dan = Tx.insert "customer" (Changeset.insert (New.customer "Dan" "dan@example.com"))
                // let! gives the next step what this one stored: Dan, with his new key.
                let! daniel = Tx.update "rename" (dan |> Changeset.edit (fun c -> { c with Name = "Daniel" }))
                return daniel.Name
            }

        let! r = k.Repo.Transact rename
        ok r |> shouldBe __
    })

In [ ]:
koanTask "Steps compose with let!" (fun k ->
    task {
        let signUp =
            transaction {
                let! dan = Tx.insert "customer" (Changeset.insert (New.customer "Dan" "dan@example.com"))
                let! order = Tx.insert "order" (Changeset.insert (New.order (Some dan.Id) Draft 1m))
                return order.CustomerId = Some dan.Id
            }

        let! r = k.Repo.Transact signUp
        ok r |> shouldBe __
        let! n = customers k
        n |> shouldBe __
    })

In [ ]:
koanTask "A failure names its step" (fun k ->
    task {
        let signUp =
            transaction {
                let! dan = Tx.insert "customer" (Changeset.insert (New.customer "Dan" "dan@example.com"))
                // A negative total breaks orders_total_check.
                let! _ = Tx.insert "order" (Changeset.insert (New.order (Some dan.Id) Draft -1m))
                return ()
            }

        let! r = k.Repo.Transact signUp
        let e = err r
        e.Step |> shouldBe __
        e.RolledBack |> shouldBe __
        let! n = customers k
        n |> shouldBe __
    })

In [ ]:
koanTask "An empty transaction is fine" (fun k ->
    task {
        let! r = k.Repo.Transact(transaction { return 42 })
        r |> shouldBe __
    })

In [ ]:
koan "Retries are opt-in" (fun _ ->
    RetryPolicy.None.MaxAttempts |> shouldBe __
    (RetryPolicy.OnSerializationFailure 3).MaxAttempts |> shouldBe __)

In [ ]:
koanTask "Custom steps use the transaction" (fun k ->
    task {
        let audit =
            transaction {
                let! _ =
                    Tx.run "audit" (fun ctx ->
                        task {
                            use cmd = ctx.CreateCommand()
                            cmd.CommandText <- "insert into products(sku, name) values ('AUDIT', 'Audit')"
                            let! _ = cmd.ExecuteNonQueryAsync(ctx.CancellationToken)
                            return (Ok(): Result<unit, string>)
                        })

                return! Tx.fail "stop" "changed my mind"
            }

        let! r = k.Repo.Transact audit
        (err r).Step |> shouldBe __
        let! audited = k.Repo.Count(plasm { for p in Db.products do where (p.Sku = "AUDIT") })
        ok audited |> shouldBe __
    })

In [ ]:
koanTask "Effects outside the database are not rolled back" (fun k ->
    task {
        let emailsSent = ref 0

        let tx =
            transaction {
                let! _ =
                    Tx.run "email" (fun _ ->
                        task {
                            emailsSent.Value <- emailsSent.Value + 1
                            return (Ok(): Result<unit, string>)
                        })

                return! Tx.fail "stop" "changed my mind"
            }

        let! _ = k.Repo.Transact tx
        emailsSent.Value |> shouldBe __
    })

## About joins

Joins return combined rows from one command; left joins give the missing side as None.

In [ ]:
open Plasm
open Plasm.Koans
open Plasm.Koans.Shop

In [ ]:
koanTask "The simplest inner join" (fun k ->
    task {
        // join pairs each row with the rows of another table whose key matches (an inner join).
        // Every OrderLine has a ProductId, so every line finds its Product.
        let linesWithProducts =
            plasm {
                for l in Db.orderLines do
                    join p in Db.products on (l.ProductId = p.Id)
                    orderBy l.Id
            }

        let! rows = k.Repo.All linesWithProducts
        ok rows |> List.map (fun (_, p) -> p.Name) |> shouldBe __
    })

In [ ]:
koanTask "Selecting a pair's fields" (fun k ->
    task {
        // select can name fields from either side of the join.
        let orderOne =
            plasm {
                for l in Db.orderLines do
                    join p in Db.products on (l.ProductId = p.Id)
                    where (l.OrderId = OrderId 1L)
                    orderBy l.Id
                    select (p.Name, l.UnitPrice)
            }

        let! rows = k.Repo.All orderOne
        ok rows |> shouldBe __
    })

In [ ]:
koanTask "Counting joined rows" (fun k ->
    task {
        // A join gives one row for each matching pair, so an Order appears once per Payment.
        let paid =
            plasm {
                for o in Db.orders do
                    join p in Db.payments on (o.Id = p.OrderId)
                    orderBy p.Id
            }

        let! n = k.Repo.Count paid
        ok n |> shouldBe __
    })

In [ ]:
koanTask "A left join's missing side is None" (fun k ->
    task {
        // leftJoin keeps rows with no match; the other side is an option, None when missing.
        let fourth =
            plasm {
                for o in Db.orders do
                    leftJoin c in Db.customers on (o.CustomerId = Some c.Id)
                    where (o.Id = OrderId 4L)
            }

        let! row = k.Repo.One fourth
        let (_, customer) = (ok row).Value
        customer |> shouldBe __
    })

In [ ]:
koanTask "An inner join gives pairs" (fun k ->
    task {
        let pairs =
            plasm {
                for o in Db.orders do
                    join c in Db.customers on (o.CustomerId = Some c.Id)
                    orderBy o.Id
            }

        let! rows = k.Repo.All pairs
        ok rows |> List.map (fun (o, c) -> o.Id, c.Name) |> shouldBe __
    })

In [ ]:
koanTask "A left join keeps every row" (fun k ->
    task {
        let all =
            plasm {
                for o in Db.orders do
                    leftJoin c in Db.customers on (o.CustomerId = Some c.Id)
                    orderBy o.Id
            }

        let! rows = k.Repo.All all
        ok rows |> List.map (fun (o, c) -> o.Id, c |> Option.map (fun c -> c.Name)) |> List.last |> shouldBe __
    })

In [ ]:
koanTask "Join keys are typed" (fun k ->
    task {
        // The keys must have the same type: an optional CustomerId on the Order side.
        let pairs =
            plasm {
                for o in Db.orders do
                    join c in Db.customers on (o.CustomerId = __)
                    orderBy o.Id
            }

        let! rows = k.Repo.Count pairs
        ok rows |> shouldBe 3L
    })

In [ ]:
koanTask "Filters on joined columns run in SQL" (fun k ->
    task {
        let adas =
            plasm {
                for o in Db.orders do
                    join c in Db.customers on (o.CustomerId = Some c.Id)
                    where (c.Name = "Ada")
            }

        (k.Sql adas).Contains "WHERE" |> shouldBe __
        let! n = k.Repo.Count adas
        ok n |> shouldBe __
    })

In [ ]:
koanTask "Joined queries compose" (fun k ->
    task {
        let withCustomers = plasm { for o in Db.orders do join c in Db.customers on (o.CustomerId = Some c.Id); orderBy o.Id }
        let large (q: Query<Order * Customer>) = plasm { for (o, c) in q do where (o.Total > 20m) }
        let! rows = k.Repo.All(large withCustomers)
        ok rows |> List.map (fun (_, c) -> c.Name) |> shouldBe __
    })

## About aggregates and grouping

Aggregates are typed values computed in the database; grouping summarises without transferring rows.

In [ ]:
open Plasm
open Plasm.Koans
open Plasm.Koans.Shop

In [ ]:
koanTask "Counting rows with Agg.count" (fun k ->
    task {
        // An aggregate summarises many rows as one value, computed by the database.
        // Agg.count () is the number of rows; One reads the single row it gives.
        let! n = k.Repo.One(plasm { for o in Db.orders do where (o.Status = Published); select (Agg.count ()) })
        (ok n).Value |> shouldBe __
    })

In [ ]:
koanTask "Adding up with Agg.sum" (fun k ->
    task {
        // Agg.sum adds up a column over the rows the query matches.
        let! total = k.Repo.One(plasm { for o in Db.orders do where (o.Status = Published); select (Agg.sum o.Total) })
        (ok total).Value |> shouldBe __
    })

In [ ]:
koanTask "The largest with Agg.max" (fun k ->
    task {
        // Agg.max is the largest value: an option, because there might be no rows.
        let! largest = k.Repo.One(plasm { for o in Db.orders do select (Agg.max o.Total) })
        (ok largest).Value |> shouldBe __
    })

In [ ]:
koanTask "Grouping by one key" (fun k ->
    task {
        // groupBy puts rows with the same key in one group; select gives one row per group.
        let linesPerOrder =
            plasm {
                for l in Db.orderLines do
                    groupBy l.OrderId
                    orderBy l.OrderId
                    select (l.OrderId, Agg.count ())
            }

        let! rows = k.Repo.All linesPerOrder
        ok rows |> shouldBe __
    })

In [ ]:
koanTask "Counting a page counts the page" (fun k ->
    task {
        let! n = k.Repo.Count(plasm { for o in Db.orders do orderBy o.Id; take 2 })
        ok n |> shouldBe __
    })

In [ ]:
koanTask "Sum and average are typed" (fun k ->
    task {
        let! totals = k.Repo.One(plasm { for o in Db.orders do select (Agg.sum o.Total, Agg.avg o.Total) })
        // The four totals are 30.00, 12.50, 99.00 and 5.00.
        (ok totals).Value |> shouldBe __
    })

In [ ]:
koanTask "No rows is explicit" (fun k ->
    task {
        let! r = k.Repo.One(plasm { for o in Db.orders do where (o.Total > 1000m); select (Agg.sum o.Total, Agg.max o.Total) })
        (ok r).Value |> shouldBe __
    })

In [ ]:
koanTask "Grouping by a key" (fun k ->
    task {
        let perCustomer =
            plasm {
                for o in Db.orders do
                    groupBy o.CustomerId
                    orderBy o.CustomerId
                    select (o.CustomerId, Agg.count ())
            }

        let! rows = k.Repo.All perCustomer
        ok rows |> shouldBe __
    })

In [ ]:
koanTask "Filtering groups" (fun k ->
    task {
        let repeat =
            plasm {
                for o in Db.orders do
                    groupBy o.CustomerId
                    having (Agg.count () > 1L)
                    select (o.CustomerId, Agg.count ())
            }

        let! rows = k.Repo.All repeat
        ok rows |> shouldBe __
    })

In [ ]:
koanTask "Grouping mistakes are refused" (fun k ->
    task {
        // Status is neither grouped nor aggregated.
        let wrong = plasm { for o in Db.orders do groupBy o.CustomerId; select (o.Status, Agg.count ()) }
        let! r = k.Repo.All wrong
        Result.isError r |> shouldBe __
        k.Commands.Length |> shouldBe __
    })

## About many-to-many and has-one

Join-table relationships and one-to-one relationships load and filter like any other; link writes touch only the join table.

In [ ]:
open Plasm
open Plasm.Koans
open Plasm.Koans.Shop

let names (tags: Assoc<Tag list>) = tags |> Assoc.get |> List.map (fun t -> t.Name) |> List.sort

let article (k: KoanContext) (id: int64) =
    task {
        let! a = k.Repo.One(plasm { for a in Db.articles do where (a.Id = ArticleId id); preload a.Tags })
        return (ok a).Value
    }

let tag id name = { Id = TagId id; Name = name }

In [ ]:
koanTask "Tags wait to be asked for" (fun k ->
    task {
        // Articles and Tags are many-to-many: the article_tags table links them in pairs.
        // Without a preload, the Tags are not read at all.
        let! intro = k.Repo.One(plasm { for a in Db.articles do where (a.Id = ArticleId 1L) })
        (ok intro).Value.Tags |> shouldBe __
    })

In [ ]:
koanTask "Reading an article's tags" (fun k ->
    task {
        // The helper `article` (top of this file) reads one Article with `preload a.Tags`.
        // In the starting data, article 2 ('Deep dive') is linked to tags 1 and 3.
        let! deepDive = article k 2L
        deepDive.Tags |> names |> shouldBe __
    })

In [ ]:
koanTask "An account's profile is an option" (fun k ->
    task {
        // An Account has at most one Profile (has-one), so a loaded Profile is an option.
        let! ada = k.Repo.One(plasm { for a in Db.accounts do where (a.Name = "ada"); preload a.Profile })
        (ok ada).Value.Profile |> Assoc.get |> Option.map (fun p -> p.Bio) |> shouldBe __
    })

In [ ]:
koanTask "An account without a profile" (fun k ->
    task {
        // In the starting data, only account 1 ('ada') has a profile.
        let! bob = k.Repo.One(plasm { for a in Db.accounts do where (a.Name = "bob"); preload a.Profile })
        (ok bob).Value.Profile |> shouldBe __
    })

In [ ]:
koanTask "Tags load through the join table" (fun k ->
    task {
        let! articles = k.Repo.All(plasm { for a in Db.articles do orderBy a.Id; preload a.Tags })
        (ok articles).Head.Tags |> names |> shouldBe __
        k.Commands.Length |> shouldBe __
    })

In [ ]:
koanTask "No tags is an empty, loaded list" (fun k ->
    task {
        let! empty = article k 3L
        empty.Tags |> shouldBe __
    })

In [ ]:
koanTask "Filtering through links" (fun k ->
    task {
        let fsharp = plasm { for a in Db.articles do where (Rel.any (fun (t: Tag) -> t.Name = "fsharp") a.Tags); orderBy a.Id }
        let! rows = k.Repo.All fsharp
        ok rows |> List.map (fun a -> a.Title) |> shouldBe __
    })

In [ ]:
koanTask "Replacing links" (fun k ->
    task {
        let! intro = article k 1L
        let! updated = k.Repo.Update(Changeset.update intro |> Changeset.putLinks ((fun a -> a.Tags), [ tag 4L "ops" ]))
        (ok updated).Tags |> names |> shouldBe __
        let! tags = k.Repo.Count(Query.from Db.tags)
        ok tags |> shouldBe __
    })

In [ ]:
koanTask "Adding and removing one link" (fun k ->
    task {
        let! intro = article k 1L

        let change =
            Changeset.update intro
            |> Changeset.addLinks ((fun a -> a.Tags), [ tag 3L "draft" ])
            |> Changeset.removeLinks ((fun a -> a.Tags), [ tag 2L "sql" ])

        let! updated = k.Repo.Update change
        (ok updated).Tags |> shouldBe __
        let! reloaded = article k 1L
        reloaded.Tags |> names |> shouldBe __
    })

In [ ]:
koanTask "At most one related record" (fun k ->
    task {
        let! accounts = k.Repo.All(plasm { for a in Db.accounts do orderBy a.Id; preload a.Profile })
        ok accounts |> List.map (fun a -> a.Profile |> Assoc.get |> Option.map (fun p -> p.Bio)) |> shouldBe __
    })

In [ ]:
koanTask "Filtering through has-one" (fun k ->
    task {
        let writers = plasm { for a in Db.accounts do where (Rel.exists (fun p -> p.Bio.Contains "compiler") a.Profile) }
        let! rows = k.Repo.All writers
        ok rows |> List.map (fun a -> a.Name) |> shouldBe __
    })

## About bulk writes

Bulk updates and deletes are driven by typed queries; inserts take many records in one command.

In [ ]:
open Plasm
open Plasm.Koans
open Plasm.Koans.Shop

In [ ]:
koanTask "A bulk update returns a count" (fun k ->
    task {
        // A bulk update changes every row a query matches, in one SQL command.
        // The function says what each matching row becomes.
        let gizmos = plasm { for p in Db.products do where (p.Sku = "GIZMO") }
        let! changed = k.Repo.Execute(Bulk.update (gizmos, fun p -> { p with Name = "Gizmo Mark II" }))
        // The result is how many rows were changed.
        ok changed |> shouldBe __
    })

In [ ]:
koanTask "A bulk update can match nothing" (fun k ->
    task {
        let sprockets = plasm { for p in Db.products do where (p.Sku = "SPROCKET") }
        let! changed = k.Repo.Execute(Bulk.update (sprockets, fun p -> { p with Name = "Sprocket" }))
        ok changed |> shouldBe __
    })

In [ ]:
koanTask "A bulk delete returns a count" (fun k ->
    task {
        // A bulk delete removes every row a query matches, and returns how many.
        let ofOrder1 = plasm { for p in Db.payments do where (p.OrderId = OrderId 1L) }
        let! removed = k.Repo.Execute(Bulk.delete ofOrder1)
        ok removed |> shouldBe __
    })

In [ ]:
koanTask "Deleting every payment" (fun k ->
    task {
        // Query.from with no filter matches every row of the table.
        let! removed = k.Repo.Execute(Bulk.delete (Query.from Db.payments))
        ok removed |> shouldBe __
    })

In [ ]:
koanTask "Updating what a query matches" (fun k ->
    task {
        let drafts = plasm { for o in Db.orders do where (o.Status = Draft) }
        let! changed = k.Repo.Execute(Bulk.update (drafts, fun o -> { o with Status = Published }))
        ok changed |> shouldBe __
        let! published = k.Repo.Count(plasm { for o in Db.orders do where (o.Status = Published) })
        ok published |> shouldBe __
    })

In [ ]:
koanTask "Updating from the current row" (fun k ->
    task {
        let! _ = k.Repo.Execute(Bulk.update (Query.from Db.orders, fun o -> { o with Total = o.Total * 2m }))
        let! total = k.Repo.One(plasm { for o in Db.orders do select (Agg.sum o.Total) })
        (ok total).Value |> shouldBe __
    })

In [ ]:
koanTask "Deleting what a query matches" (fun k ->
    task {
        let failed = plasm { for p in Db.payments do where (p.Status = Failed) }
        let! removed = k.Repo.Execute(Bulk.delete failed)
        ok removed |> shouldBe __
        let! left = k.Repo.Count(Query.from Db.payments)
        ok left |> shouldBe __
    })

In [ ]:
koanTask "Inserting many records" (fun k ->
    task {
        let parts = [ New.product "BOLT" "Bolt"; New.product "NUT" "Nut"; New.product "WASHER" "Washer" ]
        let! inserted = k.Repo.Execute(Bulk.insert parts)
        ok inserted |> List.map (fun p -> p.Sku) |> shouldBe __
        k.Commands.Length |> shouldBe __
    })

In [ ]:
koanTask "Inserting only what is missing" (fun k ->
    task {
        let catalogue = [ New.product "WIDGET" "Widget"; New.product "SPROCKET" "Sprocket" ]
        let! inserted = k.Repo.Execute(Bulk.insertMissing (catalogue, fun p -> p.Sku))
        ok inserted |> List.map (fun p -> p.Sku) |> shouldBe __
    })

In [ ]:
koanTask "Bulk writes can be inspected" (fun k ->
    task {
        let plan = k.Repo.Explain(Bulk.delete (plasm { for p in Db.payments do where (p.Status = Failed) }))
        (ok plan).Sql.StartsWith "DELETE FROM \"payments\"" |> shouldBe __
        k.Commands.Length |> shouldBe __
    })

## About hand-written SQL

When a query is easier in SQL, write SQL; interpolated values are always parameters and rows still come back typed.

In [ ]:
open Plasm
open Plasm.Koans
open Plasm.Koans.Shop

In [ ]:
koanTask "A hand-written SELECT returns records" (fun k ->
    task {
        // repo.Sql<'R> runs SQL you write ($"..." is an interpolated string) and reads
        // each row into an 'R. Columns are matched to fields by name.
        let! products = k.Repo.Sql<Product>($"SELECT * FROM products ORDER BY id")
        ok products |> List.map (fun p -> p.Sku) |> shouldBe __
    })

In [ ]:
koanTask "No matching rows is an empty list" (fun k ->
    task {
        let! products = k.Repo.Sql<Product>($"SELECT * FROM products WHERE sku = 'SPROCKET'")
        ok products |> shouldBe __
    })

In [ ]:
koanTask "SQL can return a single number" (fun k ->
    task {
        // A row with one column can be read as that column's value.
        let! counts = k.Repo.Sql<int64>($"SELECT count(*) FROM products")
        ok counts |> shouldBe __
    })

In [ ]:
koanTask "SQL can return pairs" (fun k ->
    task {
        // A row with two columns can be read as a tuple: the first column, then the second.
        let! pairs = k.Repo.Sql<string * string>($"SELECT sku, name FROM products ORDER BY id")
        ok pairs |> List.head |> shouldBe __
    })

In [ ]:
koanTask "An interpolated value becomes a parameter" (fun k ->
    task {
        // A parameter is a value sent beside the SQL text, never pasted into it.
        let sku = "GADGET"
        let! found = k.Repo.Sql<Product>($"SELECT * FROM products WHERE sku = {sku}")
        ok found |> List.map (fun p -> p.Name) |> shouldBe __
        // k.Commands is the SQL text this koan has sent, in order.
        (List.last k.Commands).Contains "GADGET" |> shouldBe __
    })

In [ ]:
koanTask "SQL can return mapped records" (fun k ->
    task {
        let minimum = 50m
        let! big = k.Repo.Sql<Order>($"SELECT * FROM orders WHERE total >= {minimum} ORDER BY id")
        ok big |> List.map (fun o -> o.Id) |> shouldBe __
    })

In [ ]:
koanTask "SQL can return tuples and scalars" (fun k ->
    task {
        let! names = k.Repo.Sql<string>($"SELECT name FROM customers ORDER BY name")
        ok names |> shouldBe __
        let! perStatus = k.Repo.Sql<string * int64>($"SELECT status, count(*) FROM orders GROUP BY status ORDER BY status")
        ok perStatus |> List.head |> shouldBe __
    })

In [ ]:
koanTask "Interpolation is parameters" (fun k ->
    task {
        let name = "Ada'; drop table customers; --"
        let! found = k.Repo.Sql<Customer>($"SELECT * FROM customers WHERE name = {name}")
        ok found |> shouldBe __
        (List.last k.Commands).Contains "@p0" |> shouldBe __
    })

In [ ]:
koanTask "A fragment inside a typed query" (fun k ->
    task {
        let startingWithA = plasm { for c in Db.customers do where (Raw.fragment<bool> "{0} ILIKE {1}" (c.Name, "a%")) }
        let! rows = k.Repo.All startingWithA
        ok rows |> List.map (fun c -> c.Name) |> shouldBe __
    })

In [ ]:
koanTask "Commands return affected rows" (fun k ->
    task {
        let! n = k.Repo.ExecuteSql $"update orders set status = 'cancelled' where status = 'published'"
        ok n |> shouldBe __
    })

## About prepared queries

A hot query can be built and planned once and bound to new arguments on each call.

In [ ]:
open System
open Plasm
open Plasm.Koans
open Plasm.Koans.Shop

let ordersOf =
    Query.prepare (fun (customer: Arg<CustomerId>) ->
        plasm {
            for o in Db.orders do
                where (o.CustomerId = Some customer.Value)
                orderBy o.Id
        })

/// A prepared query: built once, then called with a SKU to give the query for that SKU.
/// An Arg<string> stands for the value given at each call; `.Value` reads it inside the query.
let productsWithSku =
    Query.prepare (fun (sku: Arg<string>) -> plasm { for p in Db.products do where (p.Sku = sku.Value) })

let names (r: Result<Product list, DbError>) = ok r |> List.map (fun p -> p.Name)

In [ ]:
koanTask "A prepared query takes an argument" (fun k ->
    task {
        // productsWithSku (top of this file) is a function: give it a SKU, get a query.
        let! gizmos = k.Repo.All(productsWithSku "GIZMO")
        names gizmos |> shouldBe __
    })

In [ ]:
koanTask "One prepared query, two values" (fun k ->
    task {
        let! widgets = k.Repo.All(productsWithSku "WIDGET")
        let! gadgets = k.Repo.All(productsWithSku "GADGET")
        names widgets |> shouldBe __
        names gadgets |> shouldBe __
    })

In [ ]:
koanTask "A prepared query can be counted" (fun k ->
    task {
        // ordersOf (top of this file) is prepared with a CustomerId. Customer 3 is Cy.
        let! cys = k.Repo.Count(ordersOf (CustomerId 3L))
        ok cys |> shouldBe __
    })

In [ ]:
koanTask "Built once, run many times" (fun k ->
    task {
        let! adas = k.Repo.Count(ordersOf (CustomerId 1L))
        let! bobs = k.Repo.Count(ordersOf (CustomerId 2L))
        (ok adas, ok bobs) |> shouldBe __
        // Did the two counts send the same SQL text?
        (k.Commands.[0] = k.Commands.[1]) |> shouldBe __
    })

In [ ]:
koan "Arguments are parameters" (fun k ->
    let sql = k.Sql(ordersOf (CustomerId 42L))
    sql.Contains "42" |> shouldBe __)

In [ ]:
koanTask "Paging by arguments" (fun k ->
    task {
        let page =
            Query.prepare2 (fun (size: Arg<int>) (offset: Arg<int>) ->
                plasm {
                    for o in Db.orders do
                        orderBy o.Id
                        skip offset.Value
                        take size.Value
                })

        let! second = k.Repo.All(page 2 2)
        ok second |> List.map (fun o -> o.Id) |> shouldBe __
    })

In [ ]:
koanTask "Prepared queries still compose" (fun k ->
    task {
        let publishedOf c = plasm { for o in ordersOf c do where (o.Status = Published) }
        let! n = k.Repo.Count(publishedOf (CustomerId 1L))
        ok n |> shouldBe __
    })

In [ ]:
koan "Arguments are read only inside the query" (fun _ ->
    let refused =
        try
            Query.prepare (fun (customer: Arg<CustomerId>) ->
                let id = customer.Value // read while the query is being built
                plasm { for o in Db.orders do where (o.CustomerId = Some id) })
            |> ignore

            false
        with :? InvalidOperationException ->
            true

    refused |> shouldBe __)

In [ ]:
koanTask "Prepared load queries" (fun k ->
    task {
        let paymentsWith =
            Query.prepare (fun (status: Arg<PaymentStatus>) -> plasm { for p in Db.payments do where (p.Status = status.Value) })

        let! orders = k.Repo.All(plasm { for o in Db.orders do where (o.Id = OrderId 3L); preload o.Payments (paymentsWith Failed) })
        (ok orders).Head.Payments |> Assoc.get |> List.map (fun p -> p.Status) |> shouldBe __
    })

## About failures

Expected failures are values you can match on; Plasm's error cases are always qualified, so your domain keeps its own names.

In [ ]:
open System.Threading.Tasks
open Plasm
open Plasm.Koans
open Plasm.Koans.Shop

/// Your own union may use the names Plasm uses for its errors.
type Problem =
    | Database
    | Translation

/// A domain error a transaction step can fail with.
type StockError = OutOfStock of sku: string

In [ ]:
koanTask "Nothing found is not a failure" (fun k ->
    task {
        // Every repository call returns a Result: Ok with the value, or Error with what went wrong.
        let! r = k.Repo.One(plasm { for c in Db.customers do where (c.Name = "Zed") })
        r |> shouldBe __
    })

In [ ]:
koanTask "A failure is an Error you can match on" (fun k ->
    task {
        // customers.email is unique (see the schema), and Ada already has this email.
        let! r = k.Repo.Insert(Changeset.insert (New.customer "Ada again" "ada@example.com"))

        let outcome =
            match r with
            | Ok _ -> "saved"
            | Error _ -> "refused"

        outcome |> shouldBe __
    })

In [ ]:
koanTask "A failed write says what kind of failure" (fun k ->
    task {
        // WriteError.Failed is a failure reported by the database. Its Kind is a DbFailureKind:
        // UniqueViolation, ForeignKeyViolation, CheckViolation, NotNullViolation, and others.
        let! r = k.Repo.Insert(Changeset.insert (New.customer "Ada again" "ada@example.com"))

        match r with
        | Error(WriteError.Failed f) -> f.Kind |> shouldBe __
        | other -> failwithf "expected a database failure, got %A" other
    })

In [ ]:
koanTask "A unique violation names its constraint" (fun k ->
    task {
        let! r = k.Repo.Insert(Changeset.insert (New.customer "Ada again" "ada@example.com"))

        match r with
        // Constraint is the name of the database constraint that refused the row, if any.
        | Error(WriteError.Failed f) -> f.Constraint |> shouldBe __
        | other -> failwithf "expected a database failure, got %A" other
    })

In [ ]:
koanTask "Failures are values" (fun k ->
    task {
        let kept = ref Unchecked.defaultof<Repo>

        do!
            k.Repo.Sandbox(fun inner ->
                kept.Value <- inner
                Task.FromResult())

        // The sandbox has ended; its repository returns an error rather than throwing.
        let! r = kept.Value.All(Query.from Db.customers)

        match r with
        | Error(DbError.Database f) -> f.Kind |> shouldBe __
        | other -> failwithf "expected a database error, got %A" other
    })

In [ ]:
koan "Error cases are qualified" (fun _ ->
    let classify (e: DbError) =
        match e with
        | DbError.Database _ -> Database
        | _ -> Translation

    classify (DbError.Translation(TranslationError.InvalidQuery "example")) |> shouldBe __)

In [ ]:
koanTask "Unreadable rows are named" (fun k ->
    task {
        let! _ = k.Repo.ExecuteSql $"update orders set status = 'lost' where id = 1"
        let! r = k.Repo.All(Query.from Db.orders)

        match r with
        | Error(DbError.Materialization m) -> (m.Model, m.Property) |> shouldBe __
        | other -> failwithf "expected a materialisation error, got %A" other
    })

In [ ]:
koanTask "SQL mismatches name the column" (fun k ->
    task {
        // Customer needs an email column, and this query does not return one.
        let! r = k.Repo.Sql<Customer>($"SELECT id, name FROM customers")

        match r with
        | Error e -> (string e).Contains "Email" |> shouldBe __
        | Ok rows -> failwithf "expected an error, got %A" rows
    })

In [ ]:
koanTask "Too many rows is about the data" (fun k ->
    task {
        let! r = k.Repo.One(plasm { for o in Db.orders do where (o.Status = Published) })
        r |> shouldBe __
    })

In [ ]:
koanTask "Rejections keep their type" (fun k ->
    task {
        let! r = k.Repo.Transact(transaction { return! Tx.fail "check stock" (OutOfStock "GIZMO") })

        match (err r).Failure with
        | StepFailure.Rejected rejection -> rejection.As<StockError>() |> shouldBe __
        | other -> failwithf "expected a rejection, got %A" other
    })

In [ ]:
koan "Every union is qualified" (fun _ ->
    let cs = Changeset.insert (New.customer "Eve" "eve@example.com")
    cs.Action |> shouldBe __)

## About testing with Plasm

Test against the real database inside sandboxes that always roll back, and check the model against the schema at start-up.

In [ ]:
open Plasm
open Plasm.Koans
open Plasm.Koans.Shop

let customers (repo: Repo) =
    task {
        let! n = repo.Count(Query.from Db.customers)
        return ok n
    }

In [ ]:
koanTask "A sandbox gives back its body's result" (fun k ->
    task {
        // A sandbox runs a function with its own repository and undoes all its changes when the
        // function ends. The sandbox returns whatever the function returns.
        // `customers` (top of this file) counts the Customers.
        let! counted = k.Repo.Sandbox(fun repo -> customers repo)
        counted |> shouldBe __
    })

In [ ]:
koanTask "An insert is visible inside its sandbox" (fun k ->
    task {
        let! inside =
            k.Repo.Sandbox(fun repo ->
                task {
                    let! _ = repo.Insert(Changeset.insert (New.customer "Dan" "dan@example.com"))
                    return! customers repo
                })

        inside |> shouldBe __
    })

In [ ]:
koanTask "A sandbox's insert is gone afterwards" (fun k ->
    task {
        let! _ = k.Repo.Sandbox(fun repo -> repo.Insert(Changeset.insert (New.customer "Dan" "dan@example.com")))
        // The sandbox has ended: count with the outer repository.
        let! after = customers k.Repo
        after |> shouldBe __
    })

In [ ]:
koanTask "A matching model has no schema issues" (fun k ->
    task {
        // CheckSchema compares the model (Db.model in Shop.fs) with the database's tables and
        // columns, and lists every mismatch it finds.
        let! issues = k.Repo.CheckSchema()
        ok issues |> List.length |> shouldBe __
    })

In [ ]:
koanTask "A sandbox rolls everything back" (fun k ->
    task {
        let! inside =
            k.Repo.Sandbox(fun repo ->
                task {
                    let! _ = repo.Insert(Changeset.insert (New.customer "Dan" "dan@example.com"))
                    return! customers repo
                })

        let! after = customers k.Repo
        (inside, after) |> shouldBe __
    })

In [ ]:
koanTask "Transactions keep their meaning in a sandbox" (fun k ->
    task {
        let failing =
            transaction {
                let! _ = Tx.insert "customer" (Changeset.insert (New.customer "Dan" "dan@example.com"))
                return! Tx.fail "stop" "changed my mind"
            }

        let! _ = k.Repo.Transact failing
        let! n = customers k.Repo
        n |> shouldBe __
    })

In [ ]:
koanTask "A database error does not end a sandbox" (fun k ->
    task {
        // Every koan already runs in a sandbox: k.Repo is one.
        let! duplicate = k.Repo.Insert(Changeset.insert (New.customer "Ada again" "ada@example.com"))
        Result.isError duplicate |> shouldBe __
        let! later = k.Repo.Insert(Changeset.insert (New.customer "Dan" "dan@example.com"))
        Result.isOk later |> shouldBe __
    })

In [ ]:
koanTask "The model can be checked against the database" (fun k ->
    task {
        let! issues = k.Repo.CheckSchema()
        issues |> shouldBe __
    })

In [ ]:
koanTask "Drift is reported precisely" (fun k ->
    task {
        // DDL is transactional in PostgreSQL, so the sandbox undoes this too.
        let! _ = k.Repo.ExecuteSql $"alter table orders drop column cancellation_reason"
        let! issues = k.Repo.CheckSchema()
        let issue = (ok issues).Head
        (issue.Kind, issue.Model, issue.Field) |> shouldBe __
    })

## About query pipelines

Every plasm { } operation has a pipeline form; both produce the same query.

In [ ]:
open Plasm
open Plasm.Koans
open Plasm.Koans.Shop

In [ ]:
koanTask "Filtering in a pipeline" (fun k ->
    task {
        // A pipeline builds a query step by step with |>, the pipe operator: `x |> f` is `f x`.
        // Query.where is the pipeline form of plasm's where.
        let bobs = Query.from Db.customers |> Query.where (fun c -> c.Name = "Bob")
        let! rows = k.Repo.All bobs
        ok rows |> List.map (fun c -> c.Email) |> shouldBe __
    })

In [ ]:
koanTask "Ordering in a pipeline" (fun k ->
    task {
        // Query.orderBy is the pipeline form of orderBy.
        let bySku = Query.from Db.products |> Query.orderBy (fun p -> p.Sku)
        let! rows = k.Repo.All bySku
        ok rows |> List.map (fun p -> p.Sku) |> shouldBe __
    })

In [ ]:
koanTask "Taking in a pipeline" (fun k ->
    task {
        // Query.take keeps only the first rows, in the order the pipeline gives.
        let firstTwo = Query.from Db.customers |> Query.orderBy (fun c -> c.Id) |> Query.take 2
        let! rows = k.Repo.All firstTwo
        ok rows |> List.map (fun c -> c.Name) |> shouldBe __
    })

In [ ]:
koanTask "Selecting in a pipeline" (fun k ->
    task {
        // Query.select comes last: the rows become the values it picks out.
        let names = Query.from Db.products |> Query.orderBy (fun p -> p.Id) |> Query.select (fun p -> p.Name)
        let! rows = k.Repo.All names
        ok rows |> shouldBe __
    })

In [ ]:
koan "A pipeline filter is a plasm filter" (fun k ->
    // k.Sql gives the SQL a query would send, without running it.
    let expression = plasm { for c in Db.customers do where (c.Name = "Bob") }
    let pipeline = Query.from Db.customers |> Query.where (fun c -> c.Name = "Bob")
    (k.Sql pipeline = k.Sql expression) |> shouldBe __)

In [ ]:
koanTask "A pipeline select is a plasm select" (fun k ->
    task {
        let expression = plasm { for p in Db.products do orderBy p.Id; select p.Sku }
        let pipeline = Query.from Db.products |> Query.orderBy (fun p -> p.Id) |> Query.select (fun p -> p.Sku)
        let! fromExpression = k.Repo.All expression
        let! fromPipeline = k.Repo.All pipeline
        (ok fromPipeline = ok fromExpression) |> shouldBe __
    })

In [ ]:
koan "Pipelines and expressions agree" (fun k ->
    let expression = plasm { for o in Db.orders do where (o.Status = Published); orderBy o.Id }

    let pipeline =
        Query.from Db.orders
        |> Query.where (fun o -> o.Status = Published)
        |> Query.orderBy (fun o -> o.Id)

    (k.Sql pipeline = k.Sql expression) |> shouldBe __)

In [ ]:
koanTask "Pipelines join as often as you like" (fun k ->
    task {
        let gadgetOrders =
            Query.from Db.orders
            |> Query.join (Db.orderLines, (fun o -> o.Id), (fun l -> l.OrderId))
            |> Query.join (Db.products, (fun (_, l) -> l.ProductId), (fun p -> p.Id))
            |> Query.where (fun ((_, _), p) -> p.Sku = "GADGET")
            |> Query.orderBy (fun ((o, _), _) -> o.Id)

        let! rows = k.Repo.All gadgetOrders
        ok rows |> List.map (fun ((o, l), _) -> o.Id, l.Quantity.Value) |> shouldBe __
    })